# STAGE 9B · Reimplementing Pereira et al. (MIDL 2023) on OUR data

## The question

Stage 9A showed TPR Disparity can be cut **73.3%** by thresholding alone, while the AUROC gap stayed identical to **1e-12**.

> **Does the published ADVERSARIAL method move the threshold-free gap, or does it too only move the manipulable metric?**

| if gradient reversal… | conclusion |
|---|---|
| reduces TPR disp but **not** the AUROC gap | **confirms Stage 9A** — the prior method moved only the metric |
| reduces the AUROC gap too | the method genuinely works; 9A narrows to *the metric overstates it* |

**Both outcomes are publishable.** This also removes Stage 9A's cross-dataset caveat: same data, same backbone, same split.

## The method

```
L = L_disease(θfe, θd) + L_proj(θp) − λ·L_proj(θfe)     λ = 0.1
```

The minus sign is a **gradient-reversal layer**, not a negated loss — negating the loss would also flip the sign for θp, making the adversary useless while still appearing to run.

## 🔒 `best.pt` CANNOT be harmed

| guard | |
|---|---|
| opened **read-only**, never written | ✅ |
| **SHA-256 verified** before and after | ✅ |
| all output to `checkpoints/stage9b/` | ✅ |
| hard `assert` aborts if any path resolves under `stage5/` | ✅ |

**If Stage 9B fails completely, Stages 1–5 are untouched.**

---
# 0 · Config & safety

**Run with `SMOKE_TEST = True` FIRST.** ~4 min, ~0.15 CU. It exercises every code path.

In [ ]:
import os, sys, json, time, math, hashlib, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch, torch.nn as nn

SMOKE_TEST = True        # <-- True first. ALWAYS.
EPOCHS     = 6
# Adversarial step size on theta_fe is LR_BACKBONE * LAMBDA_MAX.
# Pereira use lr 1e-4 for L_d and 1e-5 for -L_p(theta_fe), i.e. an ABSOLUTE
# adversarial step of 1e-5. Matching only their 0.1 RATIO while fine-tuning at
# 2e-5 gives 2e-6 -- 5x too weak to erase projection information in 6 epochs,
# and the run produces a non-result. These values reproduce their absolute
# adversarial step (5e-5 * 0.2 = 1e-5) while keeping the disease LR below
# theirs, which is correct for a fine-tune rather than training from scratch.
LAMBDA_MAX = 0.20
LR_BACKBONE= 5e-5
LR_ADV     = 1e-4        # fresh adversary -> normal LR
WEIGHT_DECAY, GRAD_CLIP, NUM_WORKERS = 0.05, 1.0, 2

from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print('  Drive already mounted')

PROJECT  = Path('/content/drive/MyDrive/Component_01')
IMG_ROOT = Path('/content/cardio_image_384')
TAR      = PROJECT / 'data' / 'images' / 'cardio_384.tar'
MANIFEST = PROJECT / 'training_manifest'
S5_CKPT  = PROJECT / 'checkpoints' / 'stage5' / 'best.pt'
S6CACHE  = PROJECT / 'reports' / 'stage6' / 'cache'
CKPT_DIR = PROJECT / 'checkpoints' / 'stage9b'; CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUT      = PROJECT / 'reports' / 'stage9b'; OUT.mkdir(parents=True, exist_ok=True)
LOCAL    = Path('/content/s9b'); LOCAL.mkdir(exist_ok=True)
sys.path.insert(0, str(PROJECT))

# ---- HARD SAFETY GUARDS ------------------------------------------------
S5DIR = (PROJECT / 'checkpoints' / 'stage5').resolve()
for p in (CKPT_DIR, OUT):
    r = p.resolve()
    assert r != S5DIR and S5DIR not in r.parents, 'output would collide with stage5!'
assert S5_CKPT.exists(), 'best.pt not found'
SHA_BEFORE = hashlib.sha256(S5_CKPT.read_bytes()).hexdigest()
print('  best.pt SHA-256 :', SHA_BEFORE[:40])
print('  outputs ->', CKPT_DIR)
print('  stage5 dir is NOT an output target: OK')

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEV == 'cuda', 'select an L4 GPU: Runtime > Change runtime type'
import subprocess
print(' ', subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
print('  mode:', 'SMOKE TEST' if SMOKE_TEST else 'FULL RUN')

---
# 0b · Stage the images

~7–11 min. Skipped automatically if already staged, so **Restart session** is free — only *Disconnect and delete runtime* wipes them.

In [ ]:
if not IMG_ROOT.exists():
    import shutil
    t0 = time.time(); lt = Path('/content/cardio_384.tar')
    if not lt.exists():
        assert TAR.exists(), 'tar not found on Drive: ' + str(TAR)
        print('  copying %.1f GB off Drive...' % (TAR.stat().st_size / 1e9))
        shutil.copy(TAR, lt)
        print('  copied in %.1f min' % ((time.time() - t0) / 60))
    print('  extracting...')
    subprocess.run(['tar', '-xf', str(lt), '-C', '/content'], check=True)
    print('  extracted, total %.1f min' % ((time.time() - t0) / 60))
    try: lt.unlink()
    except OSError: pass
if not IMG_ROOT.exists():
    cand = [p for p in Path('/content').glob('*') if p.is_dir() and (p/'test').is_dir()]
    assert cand, 'extraction produced no directory containing test/'
    IMG_ROOT = cand[0]; print('  IMG_ROOT corrected ->', IMG_ROOT)
print('  images staged:', IMG_ROOT, IMG_ROOT.exists())

---
# 1 · Gate: self-tests

**84 tests across three modules.** If any fail, stop — nothing below is trustworthy.

In [ ]:
import stage6_acr as acr, stage9_fairness as s9, stage9b_gradrev as s9b
from cxr_transforms import build_transform
tot = 0
for nm, mod in (('stage6_acr', acr), ('stage9_fairness', s9), ('stage9b_gradrev', s9b)):
    p, f = mod._selftest(verbose=False)
    print('  %-18s %3d passed  %d failed' % (nm, p, f))
    assert f == 0, nm + ' self-test FAILED'
    tot += p
print('\n  ALL %d TESTS PASSED' % tot)

---
# 2 · Data & the frozen baseline

The baseline row comes from Stage 6's **cached** probabilities — no recompute, and it guarantees the comparison uses the exact numbers already reported.

In [ ]:
PATH = s9b.PATHOLOGIES
cfg  = json.loads((MANIFEST / 'manifest_config.json').read_text())
man  = {s: pd.read_csv(MANIFEST / ('manifest_' + s + '.csv'), low_memory=False)
        for s in ('train', 'val', 'test')}
if SMOKE_TEST:
    man = {s: d.head(320).copy() for s, d in man.items()}
for s, d in man.items():
    print('  %-5s n=%-6d AP=%-5d PA=%d' % (s, len(d),
          (d.view == 'AP').sum(), (d.view == 'PA').sum()))

pw = torch.tensor([min(cfg['pos_weight'][k], 8.0) for k in PATH], dtype=torch.float32)
print('\n  pos_weight (clamped 8):', [round(float(v), 2) for v in pw])

# frozen baseline from the Stage 6 cache
BASE = None
if not SMOKE_TEST and (S6CACHE / 'probs_test.npy').exists():
    bp = pd.DataFrame(np.load(S6CACHE / 'probs_test.npy'), columns=PATH)
    bl = man['test'][PATH].astype(int).reset_index(drop=True)
    bap = (man['test'].view == 'AP').to_numpy()
    if len(bp) == len(bl):
        BASE = dict(
            auroc=float(np.nanmean([acr.auroc(bl[k], bp[k]) for k in PATH])),
            gap=float(np.nanmean([acr.auroc(bl[k][~bap], bp[k][~bap])
                                  - acr.auroc(bl[k][bap], bp[k][bap]) for k in PATH])),
            probs=bp)
        print('  BASELINE (best.pt): AUROC %.4f   AUROC gap %.4f'
              % (BASE['auroc'], BASE['gap']))
assert SMOKE_TEST or BASE is not None, 'Stage 6 cache missing -- run Stage 6 first'

---
# 3 · Model + automatic batch-size probe

The probe runs a **real forward+backward**, not a guess, so OOM cannot appear hours in.

In [ ]:
ck = torch.load(S5_CKPT, map_location='cpu', weights_only=False)   # READ ONLY
assert list(ck.get('pathologies', PATH)) == list(PATH), 'pathology ORDER differs!'
model = s9b.CXRGradRev(len(PATH))
print('  load_stage5:', model.load_stage5(ck, use_ema=True))
model = model.to(DEV).to(memory_format=torch.channels_last)
del ck

crit_d = s9b.WeightedBCE(pw).to(DEV)

def _probe(bs):
    try:
        model.train(); model.zero_grad(set_to_none=True)
        x = torch.randn(bs, 3, 384, 384, device=DEV).to(memory_format=torch.channels_last)
        y = torch.randint(0, 2, (bs, len(PATH)), device=DEV).float()
        a = torch.randint(0, 2, (bs,), device=DEV).float()
        with torch.autocast('cuda', dtype=torch.bfloat16):
            d, p = model(x, y, lambd=0.1)
            loss = crit_d(d.float(), y, torch.ones_like(y)) + s9b.projection_loss(p.float(), a)
        loss.backward(); model.zero_grad(set_to_none=True)
        del x, y, a, d, p, loss; torch.cuda.empty_cache(); return True
    except torch.cuda.OutOfMemoryError:
        model.zero_grad(set_to_none=True); torch.cuda.empty_cache(); return False

BATCH = 8
for b in (48, 40, 32, 24, 16, 8):
    if _probe(b): BATCH = b; break
print('  BATCH =', BATCH, '(largest that survives a real fwd+bwd)')

TF_TR, TF_EV = build_transform('train'), build_transform('test')
from torch.utils.data import DataLoader
def loader(split, train):
    ds = s9b.CXRDataset(man[split], IMG_ROOT, TF_TR if train else TF_EV, PATH)
    return DataLoader(ds, batch_size=BATCH if train else BATCH * 2, shuffle=train,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=train,
                      persistent_workers=NUM_WORKERS > 0)
dl = {s: loader(s, s == 'train') for s in ('train', 'val', 'test')}

STEPS = max(len(dl['train']), 1) * EPOCHS
EMA_DECAY = s9b.ema_decay_for(STEPS)
print('  steps=%d  ema_decay=%.5f' % (STEPS, EMA_DECAY))
opt = torch.optim.AdamW(model.param_groups(LR_BACKBONE, LR_ADV),
                        weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(STEPS, 1))
ema = s9b.EMA(model, EMA_DECAY)

---
# 4 · Evaluation helper

Reports the **threshold-free** AUROC gap (what matters) *and* Pereira's δ (what they optimised).

In [ ]:
@torch.no_grad()
def evaluate(split):
    model.eval()
    D, Y, A, PJ = [], [], [], []
    for x, y, w, a in dl[split]:
        x = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            d, p = model(x, None, lambd=0.0)      # label-free, Pereira protocol
        D.append(torch.sigmoid(d.float()).cpu().numpy())
        PJ.append(torch.sigmoid(p.float()).cpu().numpy())
        Y.append(y.numpy()); A.append(a.numpy())
    P = pd.DataFrame(np.concatenate(D), columns=PATH)
    L = pd.DataFrame(np.concatenate(Y).astype(int), columns=PATH)
    ap = np.concatenate(A) > 0.5
    proj = np.concatenate(PJ)
    au, gap, td = [], [], []
    for k in PATH:
        y_, p_ = L[k].to_numpy(), P[k].to_numpy()
        au.append(acr.auroc(y_, p_))
        gap.append(acr.auroc(y_[~ap], p_[~ap]) - acr.auroc(y_[ap], p_[ap]))
        t = s9.best_f1_threshold(y_, p_)
        ta, _ = s9.tpr_fpr_at(y_[ap], p_[ap], t)
        tp, _ = s9.tpr_fpr_at(y_[~ap], p_[~ap], t)
        td.append(abs(ta - tp))
    m = dict(auroc=float(np.nanmean(au)), auroc_gap=float(np.nanmean(gap)),
             tpr_disp=float(np.nanmean(td)),
             proj_auc=float(acr.auroc(ap.astype(int), proj)))
    m['delta'] = 100 * m['auroc'] - 100 * m['tpr_disp']
    model.train()
    return m, P, L, ap

m0, _, _, _ = evaluate('val')
print('  epoch 0 (loaded best.pt, fresh adversary):')
print('    AUROC %.4f | gap %.4f | TPRdisp %.4f | projAUC %.4f | delta %.2f'
      % (m0['auroc'], m0['auroc_gap'], m0['tpr_disp'], m0['proj_auc'], m0['delta']))

---
# 5 · Training ★

Checkpoint selection on **highest δ on validation** — faithful to Pereira, who note that adversarial losses are non-monotonic so a loss-based criterion is unusable.

Every epoch is saved to Drive and the run **auto-resumes** after a disconnect.

In [ ]:
from tqdm.auto import tqdm
RESUME = CKPT_DIR / ('last_smoke.pt' if SMOKE_TEST else 'last.pt')
BEST   = CKPT_DIR / ('best_smoke.pt' if SMOKE_TEST else 'best.pt')
start_ep, gstep, best_delta, hist = 0, 0, -1e9, []
if RESUME.exists():
    r = torch.load(RESUME, map_location='cpu', weights_only=False)
    model.load_state_dict(r['model']); opt.load_state_dict(r['opt'])
    sched.load_state_dict(r['sched']); ema.shadow = r['ema']
    start_ep, gstep, best_delta, hist = r['epoch'] + 1, r['gstep'], r['best_delta'], r['hist']
    print('  RESUMED from epoch', r['epoch'])

def atomic_save(obj, path):
    tmp = path.with_suffix('.tmp')
    torch.save(obj, tmp); os.replace(tmp, path)

T0 = time.time()
for ep in range(start_ep, EPOCHS):
    model.train(); run_d = run_p = n = 0
    bar = tqdm(dl['train'], desc='epoch %d/%d' % (ep + 1, EPOCHS), leave=False)
    for x, y, w, a in bar:
        x = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
        y, w, a = y.to(DEV), w.to(DEV), a.to(DEV)
        lam = s9b.lambda_at(gstep, STEPS, LAMBDA_MAX)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            d, p = model(x, y, lambd=lam)
        ld = crit_d(d.float(), y, w)
        lp = s9b.projection_loss(p.float(), a)
        loss = ld + lp
        assert torch.isfinite(loss), 'non-finite loss at step %d -- aborting' % gstep
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step(); sched.step(); ema.update(model); gstep += 1
        run_d += ld.item(); run_p += lp.item(); n += 1
        bar.set_postfix(Ld='%.3f' % (run_d/n), Lp='%.3f' % (run_p/n), lam='%.3f' % lam)
    ema.apply(model)
    mv, _, _, _ = evaluate('val')
    ema.restore(model)
    mv['epoch'] = ep; mv['Ld'] = run_d/max(n,1); mv['Lp'] = run_p/max(n,1)
    hist.append(mv)
    print('  ep%d  Ld %.4f  Lp %.4f | AUROC %.4f  gap %.4f  TPRdisp %.4f  projAUC %.4f  delta %.2f'
          % (ep+1, mv['Ld'], mv['Lp'], mv['auroc'], mv['auroc_gap'],
             mv['tpr_disp'], mv['proj_auc'], mv['delta']))
    atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(),
                     sched=sched.state_dict(), ema=ema.shadow, epoch=ep,
                     gstep=gstep, best_delta=best_delta, hist=hist), RESUME)
    if ep >= 2 and mv['proj_auc'] > 0.80:
        print('       !! projection AUC still %.3f -- the reversal is NOT biting.' % mv['proj_auc'])
        print('          Raise LAMBDA_MAX (0.2 -> 0.4) or LR_BACKBONE, then re-run.')
        print('          A run where the adversary never bites is a null experiment.')
    if mv['delta'] > best_delta:
        best_delta = mv['delta']
        atomic_save(dict(model=model.state_dict(), ema=ema.shadow, epoch=ep,
                         val=mv, hist=hist), BEST)
        print('       ^ new best delta -> saved')
print('\n  training done in %.1f min' % ((time.time()-T0)/60))

---
# 6 · THE RESULT ★★

Baseline vs gradient reversal on the **same test set**. The decisive column is **AUROC gap** — threshold-free, and the quantity Stage 9A proved thresholding cannot touch.

In [ ]:
b = torch.load(BEST, map_location='cpu', weights_only=False)
model.load_state_dict(b['model'])
msd = model.state_dict()
for k, v in b['ema'].items():
    if k in msd: msd[k].copy_(v.to(msd[k].dtype))
model = model.to(DEV)
print('  loaded best epoch', b['epoch'] + 1)

mt, Pg, Lg, apg = evaluate('test')
print()
print('=' * 92)
print('  BASELINE vs GRADIENT REVERSAL  (same test set, same split, same backbone)')
print('=' * 92)
print('  %-28s %10s %12s %12s %11s' % ('model', 'AUROC', 'AUROC gap', 'TPR Disp', 'proj AUC'))
print('  ' + '-' * 89)
if BASE:
    bt = []
    for k in PATH:
        y_, p_ = Lg[k].to_numpy(), BASE['probs'][k].to_numpy()
        t = s9.best_f1_threshold(y_, p_)
        ta, _ = s9.tpr_fpr_at(y_[apg], p_[apg], t)
        tp, _ = s9.tpr_fpr_at(y_[~apg], p_[~apg], t)
        bt.append(abs(ta - tp))
    print('  %-28s %10.4f %12.4f %12.4f %11s'
          % ('baseline (best.pt)', BASE['auroc'], BASE['gap'], float(np.nanmean(bt)), '--'))
print('  %-28s %10.4f %12.4f %12.4f %11.4f'
      % ('gradient reversal (9B)', mt['auroc'], mt['auroc_gap'], mt['tpr_disp'], mt['proj_auc']))
print('  ' + '-' * 89)
if BASE:
    dg = mt['auroc_gap'] - BASE['gap']
    da = mt['auroc'] - BASE['auroc']
    print()
    print('  AUROC gap change : %+.4f  (negative = genuinely fairer)' % dg)
    print('  AUROC cost       : %+.4f' % da)
    print()
    if dg < -0.01:
        print('  VERDICT: gradient reversal DOES reduce the threshold-free gap.')
        print('  Stage 9A narrows to: the metric overstates, but the method works.')
    else:
        print('  VERDICT: gradient reversal does NOT meaningfully reduce the')
        print('  threshold-free gap. This CONFIRMS Stage 9A -- the published method')
        print('  moved the manipulable metric, not the underlying disparity.')

print('\n  per-epoch history')
print('  %5s %9s %10s %10s %10s %8s' % ('epoch','AUROC','gap','TPRdisp','projAUC','delta'))
for h in hist:
    print('  %5d %9.4f %10.4f %10.4f %10.4f %8.2f'
          % (h['epoch']+1, h['auroc'], h['auroc_gap'], h['tpr_disp'], h['proj_auc'], h['delta']))

---
# 7 · Save & verify `best.pt` is intact

In [ ]:
from datetime import datetime
sfx = '_smoke' if SMOKE_TEST else ''
res = dict(stage='9b', timestamp=datetime.now().isoformat(), smoke=SMOKE_TEST,
           epochs=EPOCHS, batch=BATCH, lambda_max=LAMBDA_MAX,
           lr_backbone=LR_BACKBONE, lr_adv=LR_ADV, ema_decay=EMA_DECAY,
           best_epoch=int(b['epoch']), history=hist, test_gradrev=mt,
           test_baseline=(dict(auroc=BASE['auroc'], gap=BASE['gap']) if BASE else None),
           pereira_2023=s9.PEREIRA_2023)
(OUT / ('stage9b_results' + sfx + '.json')).write_text(
    json.dumps(res, indent=2, default=float), encoding='utf-8')
np.save(OUT / ('probs_test_gradrev' + sfx + '.npy'), Pg.to_numpy().astype(np.float32))
print('  saved', OUT / ('stage9b_results' + sfx + '.json'))

SHA_AFTER = hashlib.sha256(S5_CKPT.read_bytes()).hexdigest()
print()
print('  best.pt SHA-256 before :', SHA_BEFORE[:40])
print('  best.pt SHA-256 after  :', SHA_AFTER[:40])
assert SHA_AFTER == SHA_BEFORE, 'best.pt WAS MODIFIED -- this must never happen'
print('  *** best.pt VERIFIED BYTE-IDENTICAL ***')
if SMOKE_TEST:
    print('\n  SMOKE TEST ONLY. Set SMOKE_TEST = False, restart, and re-run.')

---
# What this settles

| question | cell |
|---|---|
| Does adversarial training reduce the **threshold-free** gap? | §6 |
| At what accuracy cost? | §6 |
| Did the adversary actually work (projection AUC ↓)? | §6 |
| Is `best.pt` intact? | §7, SHA-256 |

Combined with Stage 9A, either verdict yields a complete, defensible story on **identical data** — removing the cross-dataset caveat entirely.